# Weather Pattern Analysis

## Project Overview
Analyze weather measurements across time and location to study trends, seasonality, and variable relationships.

## Learning Objectives
- Explore temporal patterns in temperature, precipitation, and humidity
- Compare weather profiles across locations
- Analyze inter-variable correlations (temperature-humidity, wind-precipitation)
- Detect long-term trends and anomalous periods

## Problem Statement
Meteorologists, urban planners, and agricultural managers need to understand weather patterns to plan operations, assess climate trends, and prepare for extreme events.

## Why This Project Matters
Weather affects every sector — agriculture, energy, transport, health. Understanding patterns and trends is essential for adaptation planning and risk management.

## Dataset Overview
Synthetic daily weather data: 4 stations × 5 years (~7,300 records) with temperature, precipitation, humidity, wind speed, and cloud cover.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
stations = {'Coastal': {'temp_mean': 18, 'temp_amp': 8, 'precip_base': 3.5},
            'Mountain': {'temp_mean': 10, 'temp_amp': 12, 'precip_base': 4.0},
            'Plains': {'temp_mean': 16, 'temp_amp': 15, 'precip_base': 2.0},
            'Urban': {'temp_mean': 19, 'temp_amp': 13, 'precip_base': 2.5}}
n_days = 365 * 5
dates = pd.date_range('2019-01-01', periods=n_days, freq='D')
rows = []
for station, params in stations.items():
    doy = dates.dayofyear
    temp = params['temp_mean'] + params['temp_amp'] * np.sin(2*np.pi*(doy-80)/365.25) + np.random.normal(0, 3, n_days)
    # Slight warming trend
    temp += np.arange(n_days) * 0.001
    precip = np.clip(np.random.exponential(params['precip_base'], n_days), 0, 50).round(1)
    # More precip in winter for coastal/mountain
    if station in ['Coastal', 'Mountain']:
        winter_boost = np.where((dates.month <= 2) | (dates.month >= 11), 1.5, 1.0)
        precip = (precip * winter_boost).round(1)
    humidity = np.clip(65 + 15*np.sin(2*np.pi*(doy-170)/365.25) - temp*0.3 + np.random.normal(0, 8, n_days), 20, 100).round(0)
    wind = np.clip(np.random.exponential(12, n_days), 0, 60).round(1)
    cloud = np.clip(np.where(precip > 2, 70 + np.random.normal(0, 10, n_days),
                              40 + np.random.normal(0, 15, n_days)), 0, 100).round(0)
    for i in range(n_days):
        rows.append({'Date': dates[i], 'Station': station, 'Temperature': round(temp[i], 1),
                     'Precipitation': precip[i], 'Humidity': humidity[i],
                     'WindSpeed': wind[i], 'CloudCover': cloud[i]})
df = pd.DataFrame(rows)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
df['Season'] = df['Month'].map({12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',
                                 6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall'})
print(f'Dataset: {df.shape}')
print(f'Stations: {df["Station"].nunique()}, Years: {df["Year"].nunique()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Records per station: {df.groupby("Station").size().to_dict()}')

## Temperature Trends by Station

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
for station in df['Station'].unique():
    sub = df[df['Station'] == station].set_index('Date')['Temperature'].rolling(30).mean()
    ax.plot(sub.index, sub.values, label=station, linewidth=1)
ax.set_title('30-Day Rolling Temperature by Station')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'temp_trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Temperature Profiles

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
pivot = df.groupby(['Station', 'Month'])['Temperature'].mean().unstack(level=0)
pivot.plot(ax=ax, marker='o')
ax.set_title('Monthly Mean Temperature by Station')
ax.set_xlabel('Month')
ax.set_ylabel('Temperature (°C)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seasonal_temp.png'), dpi=100, bbox_inches='tight')
plt.show()

## Precipitation Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('Station')['Precipitation'].mean().sort_values().plot.barh(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Mean Daily Precipitation by Station')
axes[0].set_xlabel('mm')
monthly_precip = df.groupby(['Station', 'Month'])['Precipitation'].sum().unstack(level=0) / df['Year'].nunique()
monthly_precip.plot(ax=axes[1], marker='o')
axes[1].set_title('Avg Monthly Total Precipitation')
axes[1].set_ylabel('mm')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'precipitation.png'), dpi=100, bbox_inches='tight')
plt.show()

## Variable Correlations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
num_cols = ['Temperature', 'Precipitation', 'Humidity', 'WindSpeed', 'CloudCover']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Weather Variable Correlations')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlations.png'), dpi=100, bbox_inches='tight')
plt.show()

## Long-Term Warming Trend

In [ ]:
annual_temp = df.groupby(['Year', 'Station'])['Temperature'].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 5))
annual_temp.plot(ax=ax, marker='o')
ax.set_title('Annual Mean Temperature by Station')
ax.set_ylabel('Temperature (°C)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'warming_trend.png'), dpi=100, bbox_inches='tight')
plt.show()
total_change = annual_temp.iloc[-1] - annual_temp.iloc[0]
print('Temperature change over 5 years (°C):')
print(total_change.round(2))

## Extreme Weather Days

In [ ]:
# Hot days (>35°C) and heavy rain days (>20mm)
hot_days = df[df['Temperature'] > 35].groupby('Station').size()
heavy_rain = df[df['Precipitation'] > 20].groupby('Station').size()
extremes = pd.DataFrame({'HotDays': hot_days, 'HeavyRainDays': heavy_rain}).fillna(0).astype(int)
print('Extreme weather days (5 years):')
print(extremes)

## Interpretation and Business Insight
- **Mountain station** has the widest temperature range and highest precipitation — infrastructure needs to handle both extremes
- **Urban station** shows the highest mean temperature (urban heat island effect)
- **Coastal station** has the most stable temperatures but significant winter precipitation
- **Temperature-humidity** inverse correlation confirms that hot days tend to be drier
- **Slight warming trend** is detectable even over 5 years — consistent with climate projections
- **Cloud cover** correlates with precipitation as expected — useful for solar energy planning

## Limitations
- Synthetic data with simplified climate model
- No extreme event modeling (storms, heatwaves)
- No elevation or geographic coordinates
- No air quality or pressure data
- Only 4 stations — real networks have hundreds

## How to Improve This Project
- Add extreme event analysis (heat waves, frost days, storm frequency)
- Include atmospheric pressure and air quality
- Build temperature forecasting models
- Add spatial interpolation between stations
- Compare against global climate datasets (ERA5, NOAA)

## Production Considerations
- Real-time weather monitoring dashboards
- Automated extreme weather alerts
- Integration with agriculture, energy, and transport planning
- Climate trend reporting for policy

## Common Mistakes
- Comparing stations without accounting for elevation and geography
- Drawing climate conclusions from short time series
- Ignoring station siting effects (urban heat island)
- Treating daily averages as representative of intra-day variation

## Mini Challenge / Exercises
1. Which station has the largest temperature range (max - min) over the full dataset?
2. Calculate the correlation between temperature and precipitation separately for each station.
3. How many frost days (temp < 0°C) does each station have per year on average?
4. Is the warming trend statistically significant? (Hint: linear regression on annual mean temp.)

## Final Summary / Key Takeaways
- Weather patterns show strong seasonality, geographic variation, and inter-variable correlations
- Even over 5 years, a warming trend is detectable
- Station-level differences reflect geography (coastal stability, urban heat, mountain extremes)
- Understanding weather patterns is foundational for climate adaptation and operational planning
- Multi-scale analysis (daily, monthly, annual) reveals different aspects of climate behavior